# D2.5 · Replay and forensics

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.4 · Containment at machine speed](https://spbreed.github.io/cyber-commons/lessons/D2.4.html)**.

| | |
|---|---|
| Tools used | OpenTelemetry |

## What this lesson is

**What it covers.** Replay an agent run for a regulator-grade record.

**Why a security engineer needs it.** Non-determinism as an evidentiary problem. The control it builds is: log at design time what replay will need.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Forensics on a non-deterministic actor asks a question classical forensics never had to: not just what it did, but what it saw and what it decided. If the context was not recorded, the decision cannot be reconstructed at all.

> **At CyberTravels.** Not just what the agent did, but what it saw and what it decided. If the booking note that triggered the refund was not recorded, the decision cannot be reconstructed at all. R11.

## 2 · The framework

```
   classical forensics        agentic forensics
   +------------------+       +----------------------------+
   | what did it do   |       | what did it do             |
   |                  |       | what did it SEE            |
   |                  |       | what did it DECIDE, and why|
   +------------------+       +----------------------------+

   if the context was not recorded, the decision cannot be reconstructed
```

Forensics for an agent means answering: *why did it do that?*

For ordinary software the answer is in the code. For an agent the answer is in
the run — the prompts, the tool results it saw, the model version, the sampling.
Reproduce those four and the run is deterministic. Miss one and you can describe
what happened but never demonstrate it, which matters the moment anyone
disputes your conclusion.

The field teams miss most often is the **model version**, and it is the one that
silently invalidates everything else: a provider-side upgrade changes the
behaviour with no change on your side, so a reconstruction performed after the
upgrade does not reproduce the incident that happened before it.

## 3 · The procedure, as a skill

Replay needs five inputs and the typical production run records three. The skill checks each against a real record, then replays under two later model versions — where a different action means the original decision cannot be reproduced at all.

In [ ]:
# skills/response/run-replayability-audit/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: run-replayability-audit
description: >-
  Check whether an incident run can be replayed at all — model version, seed,
  prompt, tool results, retrieved context — and what a later model version does
  to the replay. Use when forensics needs to know why the agent did what it did.
allowed-tools: Read, Grep, Glob
---

# Replay needs five things and production records three

"Why did the agent do that" is answerable only if the run can be re-run under the
conditions it ran in. That needs the model version, the sampling parameters and
seed, the exact prompt, every tool result and the retrieved context. A typical
production run records enough to see what happened and not enough to reproduce
it.

## When to use this

Before an incident, as a readiness check, and during one, to establish honestly
whether the reconstruction is possible.

## Procedure

**1 — List the five inputs and check each against a real run record.** Model
version and seed are the two usually missing, and their absence is decisive
rather than inconvenient.

**2 — Attempt the replay.** If any input is missing, say what the replay can and
cannot establish. A partial replay is still useful for the tool path and useless
for the reasoning.

**3 — Replay under later model versions.** The provider has probably upgraded.
Record whether the action changes: if it does, the original decision cannot be
reproduced at all, and that is a finding about the estate rather than about the
incident.

**4 — Cost full instrumentation.** Storage and latency for recording everything,
against the incidents where you needed it. Present both; the answer is usually to
instrument the high-tier agents only, and that is a defensible decision when the
numbers are attached.

**5 — Record what the estate has chosen.** Which agents are replayable and which
are not, so nobody assumes during an incident.

## Output contract

```json
{
  "inputs": [{"name": "str", "recorded": false}],
  "replay": {"possible": false, "establishes": ["str"], "cannot_establish": ["str"]},
  "version_drift": [{"version": "str", "action": "str", "same_as_original": false}],
  "cost": {"storage_per_run": "str", "latency_ms": 0},
  "policy": [{"tier": "str", "fully_instrumented": true}]
}
```

## Failure modes

- **Assuming replay is possible.** Check the record before promising it.
- **Replaying on the current model.** It is not the one that acted.
- **Instrumenting everything or nothing.** Tier it and write the choice down.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/response/run-replayability-audit/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/response/run-replayability-audit/scripts/run_replayability_audit.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Check whether an incident run can be replayed at all, and what a later model version does to the replay.

This is the executable half of the `run-replayability-audit` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

from dataclasses import dataclass, field

@dataclass
class Run:
    prompts: list = field(default_factory=list)
    tool_results: list = field(default_factory=list)
    model_version: str = ""
    seed: object = None

    def replayable(self):
        missing = []
        if not self.prompts:
            missing.append("prompts — cannot reconstruct what it was asked")
        if not self.tool_results:
            missing.append("tool results — the agent saw a world you cannot rebuild")
        if not self.model_version:
            missing.append("model version — a silent upgrade changes the output")
        if self.seed is None:
            missing.append("seed — sampling makes the run unrepeatable")
        return (not missing), missing

CONFIGS = {
 "fully instrumented": Run(["fix SEC-4471"], ["file contents…"], "glm-4.6@2026-07-14", 42),
 "typical production": Run(["fix SEC-4471"], ["file contents…"], "", None),
 "prompts only":       Run(["fix SEC-4471"], [], "", None),
 "actions only":       Run(),
}
for name, r in CONFIGS.items():
    ok, missing = r.replayable()
    print(f"{name:22s} replayable={ok}")
    for m in missing: print(f"      ✗ {m}")

import hashlib

def model_output(prompt, tool_result, version, seed):
    """Deterministic stand-in: output depends on ALL FOUR inputs."""
    h = hashlib.sha256(f"{prompt}|{tool_result}|{version}|{seed}".encode()).hexdigest()
    return "read_credentials" if int(h[:2], 16) % 3 == 0 else "read_source"

INCIDENT_INPUTS = ("fix SEC-4471", "billing.py: charge(card)…")

print("reproduce the incident under the ORIGINAL model version:")
orig = model_output(*INCIDENT_INPUTS, "glm-4.6@2026-07-14", 42)
print(f"   → {orig}")

print("\nreproduce it AFTER the provider upgraded (same prompts, same tool results):")
for v in ("glm-4.6@2026-08-01", "glm-4.7@2026-08-01"):
    out = model_output(*INCIDENT_INPUTS, v, 42)
    match = "reproduces" if out == orig else "DOES NOT REPRODUCE"
    print(f"   {v:22s} → {out:18s} {match}")

print("\nWithout a pinned version you cannot tell 'the agent did not do this'")
print("from 'the model that did it no longer exists'.")

COST = {
 "model version": (1,  "one string per run", "invalidates everything else if missing"),
 "seed":          (1,  "one integer per run", "makes the run repeatable"),
 "prompts":       (3,  "storage + privacy review (D1.5)", "what it was asked"),
 "tool results":  (5,  "largest volume, highest sensitivity", "what it saw"),
}
print(f"{'field':16s}{'cost':>6}  {'what it costs':38s}why it matters")
print("-" * 100)
for f, (c, cost, why) in sorted(COST.items(), key=lambda kv: kv[1][0]):
    print(f"{f:16s}{c:>6}  {cost:38s}{why}")

print("\nrecording order, by value per unit cost:")
for i, f in enumerate(sorted(COST, key=lambda k: COST[k][0]), 1):
    print(f"   {i}. {f}")

def upgrade(run, add):
    return Run(prompts=run.prompts or (["…"] if "prompts" in add else []),
               tool_results=run.tool_results or (["…"] if "tool results" in add else []),
               model_version=run.model_version or ("pinned" if "model version" in add else ""),
               seed=run.seed if run.seed is not None else (42 if "seed" in add else None))

cur = CONFIGS["typical production"]
added = set()
for f in sorted(COST, key=lambda k: COST[k][0]):
    added.add(f)
    ok, missing = upgrade(cur, added).replayable()
    print(f"\nafter adding {f:16s} replayable={ok}  still missing={len(missing)}")
assert upgrade(cur, set(COST)).replayable()[0]

## What you just proved

Only the fully instrumented run is replayable; the typical production run is missing the model version and seed. Replaying the incident under two later model versions produces a different action, so the original run does not reproduce. Adding the two cheapest fields (model version and seed) makes the typical production run replayable.

## Your turn

Add model version and seed to your agent's run records this week. Both are one field each, and together they are the difference between forensics and storytelling.

---

**Next → [D2.6 · Post-incident change surface](https://spbreed.github.io/cyber-commons/lessons/D2.6.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.5.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.5.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*